In [1]:
# input
site_anno_file = "../mbp_site_anno/data/entryId-seqNum-resi-metalResi.tsv"
domain_file = "./tmp/target_domain.tsv"
# output
sites_domain_file = "./data/entryId-tedId.tsv"

In [2]:
import pandas as pd

df_anno = pd.read_table(site_anno_file, header=None, names=["seq_id", "seq_num", "resi", "metal_resi"])
seq_id_to_seq_nums = dict(zip(df_anno["seq_id"], df_anno['seq_num']))
df_domain = pd.read_table(domain_file, header=None, names=["seq_id", "redundant_id", "entry_id", "rep_id", "domain"], na_values=[], keep_default_na=False)

In [3]:
def get_all_positions_to_domain_id(domain_id_to_ranges_str: dict):
    result = [None for _ in range(2700)]
    for k, v in domain_id_to_ranges_str.items():
        for range_str in v.split("_"):
            start, stop = range_str.split("-")
            start, stop = int(start), int(stop)
            length = stop - start + 1
            result[(start - 1): stop] = [k for _ in range(length)]

    return result

In [4]:
import tqdm
records = []
for (seq_id,), df_seq_id in tqdm.tqdm(df_domain.groupby(by=['seq_id'])):
    # seq nums with metal annotation
    seq_nums = seq_id_to_seq_nums[seq_id].split(",")
    seq_nums = [int(i) for i in seq_nums]

    all_positions_to_domain_id = get_all_positions_to_domain_id(dict(zip(df_seq_id['entry_id'], df_seq_id['domain'])))
    domain_ids = list(set([all_positions_to_domain_id[i - 1] for i in seq_nums]))
    domain_ids = [i for i in domain_ids if i is not None]

    if len(domain_ids) != 0:
        records.append({
            "seq_id": seq_id,
            "domains": ",".join(domain_ids),
        })

100%|██████████| 5677654/5677654 [10:52<00:00, 8700.47it/s]


In [5]:
pd.DataFrame(records).to_csv(sites_domain_file, sep="\t", index=None, header=None)